# pdf2ppt on Google Colab

Upload a PDF or image, convert it to editable PPTX, and download the result from Colab.

Default settings here favor a lighter Colab runtime: `tesseract` OCR and `telea` background cleanup.

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path('/content/pdf2ppt')
if not REPO_DIR.exists():
    !git clone https://github.com/wrenth04/pdf2ppt.git /content/pdf2ppt
else:
    !git -C /content/pdf2ppt pull --ff-only

os.chdir(REPO_DIR)

!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-eng tesseract-ocr-jpn tesseract-ocr-chi-sim tesseract-ocr-chi-tra libgl1 libglib2.0-0 fontconfig fonts-dejavu-core
!pip -q install -r requirements-colab.txt
!pip -q install -e . --no-deps

sys.path.insert(0, str(REPO_DIR / 'src'))
print(f'Working directory: {REPO_DIR}')

In [ ]:
from google.colab import files

uploaded = files.upload()
input_path = next(iter(uploaded.keys()))
print(f'Uploaded: {input_path}')

In [ ]:
from pathlib import Path

input_file = Path(input_path)
output_pptx = str(input_file.with_suffix('.pptx'))

# Edit these if you want different behavior.
pages = None  # Example for PDFs: '1-3,5'
ocr = 'on'
ocr_engine = 'tesseract'
deskew = True
ocr_inpaint_backend = 'telea'
debug_layout = False
image_mode = 'auto'
textbox_merge = 'off'
strict = False
ocr_lang = 'eng+jpn+chi_sim+chi_tra'
font_map = None

print(f'Output: {output_pptx}')

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, '-m', 'pdf2ppt.cli',
    input_path,
    output_pptx,
    '--ocr', ocr,
    '--ocr-engine', ocr_engine,
    '--ocr-inpaint-backend', ocr_inpaint_backend,
]
if pages:
    cmd += ['--pages', pages]
if debug_layout:
    cmd += ['--debug-layout']
if image_mode != 'auto':
    cmd += ['--image-mode', image_mode]
if textbox_merge != 'off':
    cmd += ['--textbox-merge', textbox_merge]
if strict:
    cmd += ['--strict']
if not deskew:
    cmd += ['--deskew', 'false']
if ocr_lang:
    cmd += ['--ocr-lang', ocr_lang]
if font_map:
    cmd += ['--font-map', font_map]

result = subprocess.run(cmd, cwd=str(REPO_DIR), env={**os.environ, 'PYTHONPATH': 'src'}, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
result.check_returncode()
print(f'Created {output_pptx}')

In [ ]:
from google.colab import files

files.download(output_pptx)

## Notes

- `pages` only applies to PDF input.
- If you want to use PaddleOCR, change `ocr_engine` to `paddle` after the notebook is working.
- Large PDFs may take a while in Colab.
- If OCR or font rendering looks wrong, rerun the install cell after restarting the runtime.